# 03 — Position-Specific Drill Analysis
## NFL Big Data Bowl 2027

**Purpose**: Stratify Combine tracking features by position group to evaluate movement phenotypes across drills:
- **WR / TE**: Direction-change efficiency and re-acceleration in agility drills (Short Shuttle, Three Cone).
- **Edge / DL**: First-step burst and early acceleration impulse in forty-yard dash and bag drills.
- **OL**: Lateral mirror movement and deceleration control.
- **CB / S**: Hip transitions, speed maintenance through direction change.

> **Environment**: Compatible with local and Kaggle runtimes with automated `/kaggle/input/` detection.

In [ ]:
# ── Environment & Path Setup ────────────────────────────────────────────────
import sys
import os
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

for p in [Path("."), Path(".."), Path("/kaggle/working"), Path("/kaggle/working/kaggle_notebooks")]:
    if p.exists() and str(p.resolve()) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

try:
    from src.config import (
        ensure_dirs, FIGURES_DIR, COMBINE_FEATURES_PARQUET,
        PLAYER_MASTER_PARQUET, DATA_PROCESSED, IN_KAGGLE, DATA_RAW
    )
    from src.data_loading import load_players, load_combine_results, load_combine_tracking
    from src.player_mapping import assign_position_groups, build_master_player_table
    from src.combine_features import aggregate_player_features, extract_features_from_tracking
except ImportError:
    IN_KAGGLE = Path("/kaggle").exists()
    PROJECT_ROOT = Path("/kaggle/working") if IN_KAGGLE else Path(".").resolve()
    DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
    FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
    COMBINE_FEATURES_PARQUET = DATA_PROCESSED / "combine_features.parquet"
    PLAYER_MASTER_PARQUET = DATA_PROCESSED / "player_master.parquet"
    
    raw_candidates = [
        Path("/kaggle/input/nfl-big-data-bowl-2027"),
        Path("/kaggle/input/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"),
        Path("./data/raw/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"),
        Path("./data/raw/nfl-big-data-bowl-2027"),
        Path("./data/raw"),
    ]
    DATA_RAW = raw_candidates[0]
    for c in raw_candidates:
        if (c / "players.csv").exists():
            DATA_RAW = c
            break
    if not (DATA_RAW / "players.csv").exists() and Path("/kaggle/input").exists():
        found = list(Path("/kaggle/input").glob("**/players.csv"))
        if found:
            DATA_RAW = found[0].parent
            
    def ensure_dirs():
        DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
        FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    def load_players():
        return pd.read_csv(DATA_RAW / "players.csv")
    def load_combine_results(cols=None):
        return pd.read_csv(DATA_RAW / "combine_results.csv", usecols=cols)
    def load_combine_tracking():
        return pd.read_csv(DATA_RAW / "combine_tracking.csv")
        
    def assign_position_groups(df):
        pos_col = 'nfl_position' if 'nfl_position' in df.columns else ('position' if 'position' in df.columns else None)
        if not pos_col:
            return df
        mapping = {'WR': 'WR', 'TE': 'TE', 'CB': 'CB', 'FS': 'S', 'SS': 'S', 'S': 'S', 'DE': 'Edge', 'EDGE': 'Edge', 'OLB': 'Edge', 'ILB': 'LB', 'MLB': 'LB', 'LB': 'LB', 'DT': 'DT', 'NT': 'DT', 'T': 'OL', 'OT': 'OL', 'G': 'OL', 'OG': 'OL', 'C': 'OL', 'RB': 'RB'}
        df = df.copy()
        df['position_group'] = df[pos_col].map(mapping).fillna('Other')
        return df
        
    def build_master_player_table(players, combine_results=None):
        if combine_results is not None:
            return players.merge(combine_results, on='nfl_id', how='left')
        return players

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')
ensure_dirs()
print(f"Environment: {'Kaggle' if Path('/kaggle').exists() else 'Local'}")
print(f"Data Raw: {DATA_RAW}")
print(f"Data Processed: {DATA_PROCESSED}")

## 1. Load Precomputed Movement Features

In [ ]:
features_df = None

if COMBINE_FEATURES_PARQUET.exists():
    print(f'Loading features from: {COMBINE_FEATURES_PARQUET}')
    features_df = pd.read_parquet(COMBINE_FEATURES_PARQUET)
else:
    print('Features parquet not found in processed directory. Extracting from raw tracking...')
    tracking = load_combine_tracking()
    try:
        from src.combine_features import extract_features_from_tracking
        features_df = extract_features_from_tracking(tracking)
    except Exception:
        records = []
        for (pid, dtype, att), grp in tracking.groupby(['nfl_id', 'drill_type', 'attempt']):
            s = grp['s'].values
            a = grp['a'].values if 'a' in grp.columns else np.zeros_like(s)
            dur = len(grp) * 0.1
            rec = {
                'nfl_id': pid,
                'drill_type': dtype,
                'attempt': att,
                'mean_speed': float(np.mean(s)),
                'max_speed': float(np.max(s)),
                'first_step_acceleration': float(a[:5].mean()) if len(a) >= 5 else 0.0,
                'burst_impulse_0_5s': float(s[:5].mean()) if len(s) >= 5 else 0.0,
                'burst_impulse_0_1s': float(s[:1].mean()) if len(s) >= 1 else 0.0,
                'duration_s': dur,
            }
            if 'dir' in grp.columns and len(grp) > 1:
                d = np.abs(np.diff(grp['dir'].values))
                d = np.where(d > 180, 360 - d, d)
                rec['total_direction_change'] = float(np.sum(d))
                rec['direction_change_rate'] = float(np.sum(d) / max(dur, 0.1))
            records.append(rec)
        features_df = pd.DataFrame(records)
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    features_df.to_parquet(COMBINE_FEATURES_PARQUET, index=False)

players = load_players()
players = assign_position_groups(players)
combine_results = load_combine_results(cols=None)
master = build_master_player_table(players, combine_results=combine_results)

# Aggregate player level (best attempt)
try:
    from src.combine_features import aggregate_player_features
    player_features = aggregate_player_features(features_df, agg_strategy='best_attempt')
except Exception:
    player_features = features_df.sort_values('max_speed', ascending=False).drop_duplicates(['nfl_id', 'drill_type'])

pos_map = players[['nfl_id', 'position_group']].drop_duplicates()
player_features = player_features.merge(pos_map, on='nfl_id', how='left')
print(f'Player features with position group: {player_features.shape}')
display(player_features.head())

## 2. WR / TE: Agility & Direction Change Rate

Examine direction-change rate in Short Shuttle for wide receivers.

In [ ]:
wr_shuttle = player_features[(player_features['position_group'] == 'WR') & (player_features['drill_type'] == 'SHORT_SHUTTLE')]
print(f'WRs with Short Shuttle tracking: {len(wr_shuttle)}')

if not wr_shuttle.empty and 'direction_change_rate' in wr_shuttle.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(wr_shuttle['direction_change_rate'], kde=True, color='purple')
    plt.title('WR Short Shuttle: Direction Change Rate Distribution (deg/s)')
    plt.xlabel('Direction Change Rate (deg/s)')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'wr_shuttle_dcr_hist.png', dpi=100)
    plt.show()

## 3. Edge / DL: First-Step Acceleration & Burst Impulse

Assess explosive burst metrics during early drill onset (first 0.1s - 0.5s).

In [ ]:
edge_forty = player_features[(player_features['position_group'].isin(['Edge', 'DT'])) & (player_features['drill_type'] == 'FORTY_YARD_DASH')]
print(f'Edge/DL prospects with 40-yard dash: {len(edge_forty)}')

if not edge_forty.empty and 'burst_impulse_0_5s' in edge_forty.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=edge_forty, x='position_group', y='burst_impulse_0_5s', palette='Oranges')
    plt.title('Pass Rusher Burst Impulse (First 0.5s of 40-yd Dash)')
    plt.ylabel('Mean Burst Speed (yds/s)')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'edge_dl_burst_impulse.png', dpi=100)
    plt.show()

## 4. Position Group Comparisons Across Movement Features

In [ ]:
features_to_compare = [f for f in ['max_speed', 'first_step_acceleration', 'direction_change_rate'] if f in player_features.columns]

summary = player_features.groupby('position_group')[features_to_compare].agg(['mean', 'std', 'count'])
display(summary)
print('Position drill analysis completed successfully.')